# 全分片数据并行 (Fully Sharded Data Parallel)
FSDP，全称是全分片数据并行 (Fully Sharded Data Parallel)，它是当前训练大语言模型（LLM）等超大规模模型的工业级核心技术。
简单来说，FSDP 是一种分布式训练策略，它通过智能地将模型参数、梯度和优化器状态分散存储在所有 GPU 上，从根本上解决了单块 GPU 显存不足的问题，使得在有限硬件资源下训练万亿级参数的模型成为可能。

## 🤔 为什么我们需要 FSDP？

要理解 FSDP 的价值，我们先看看它要解决的问题。

在传统的分布式训练方法中，如 DDP (Distributed Data Parallel)，每张 GPU 都必须持有一份完整的模型副本，包括模型参数、梯度和优化器状态。这就像给厨房里的每个厨师都发一份完整的、厚重的菜谱。

- DDP 的瓶颈：当模型变得极其庞大（例如数百亿、上千亿参数）时，单张 GPU 的显存会被这份“完整菜谱”直接塞满，甚至无法开始训练，导致“内存溢出 (OOM)”错误。

FSDP 的出现就是为了解决这个显存冗余问题。它借鉴了微软 DeepSpeed 的 ZeRO-3 思想，核心思路是：用通信换显存。

## 🧠 FSDP 的核心思想：分片与按需聚合

FSDP 的聪明之处在于，它不再让每张 GPU 保存完整的模型，而是将模型“切碎”，让所有 GPU 共同分担存储任务。

这个过程可以想象成一个高效的“智能图书馆”系统：
- 分片存储：图书馆（整个模型）的书（参数）被拆分，分散存放在不同的分馆（GPU）里。每个分馆只负责保管一部分书。
- 按需借阅：当某个分馆需要处理一个涉及整套丛书的复杂任务（计算某一层）时，它会临时向其他所有分馆发出请求，快速“借阅”（All-Gather）缺失的部分，凑成完整的一套。
- 立即归还：任务一完成，这套完整的丛书就会被立即“归还”（丢弃），每个分馆只保留自己原本负责的那几本。

通过这种方式，任何时刻，每张 GPU 的显存中只需要存储：
1. 自己负责的那一小部分模型参数、梯度和优化器状态（分片）。
2. 当前正在计算的那一层的完整参数（临时聚合）。

这使得单张 GPU 的显存占用理论上可以降低到 DDP 的 1/N（N 为 GPU 数量），极大地释放了显存压力。

## ⚙️ FSDP 是如何工作的？

FSDP 的工作流程主要围绕两个关键的通信操作展开，分别对应前向传播和反向传播：

1. 前向传播 (Forward Pass)
    - 当计算需要进行到模型的某一层时，所有 GPU 会执行一次 All-Gather 操作。
    - 通过这个操作，每张 GPU 都能临时获得该层的完整参数。
    - 使用该层完整参数进行计算，计算结束后，立即释放掉这些临时聚合的完整参数，只保留自己的分片。
2. 反向传播 (Backward Pass)
    - 每张 GPU 基于本地数据计算得到梯度。
    - 接着，所有 GPU 执行一次 Reduce-Scatter 操作。这个操作会先对所有 GPU 上的梯度进行聚合求平均，然后立即将结果分片，每张 GPU 只接收并保存与自己负责的参数分片相对应的那部分梯度。
    - 最后，优化器（如 Adam）也只需要更新自己负责的参数分片和对应的优化器状态分片。

| 特性 | DDP (分布式数据并行) | FSDP (全分片数据并行) |
| :--- | :--- | :--- |
| 显存占用 | 高。每张卡存储完整模型、梯度、优化器状态。 | 极低。显存占用降至约 1/N，可训练超大模型。 |
| 通信机制 | `All-Reduce`。在反向传播后同步完整梯度。 | `All-Gather` + `Reduce-Scatter`。按需通信，更复杂。 |
| 适用场景 | 中小规模模型，显存充足。 | 大规模/超大规模模型，显存是主要瓶颈。 |
| 模型上限 | 受限于单卡显存，通常在十亿(1B)参数级别。 | 理论上无上限，可支持万亿(1T+)参数模型。 |



## 🚀 实践中的关键技巧与优化

在实际使用 FSDP 时，有一些策略和注意事项可以进一步提升性能：

- 自动包装策略 (Auto Wrapping Policy)

直接对整个大模型应用 FSDP 可能不是最优的。更推荐的做法是使用自动包装策略，例如 size_based_auto_wrap_policy。它会为模型中参数量超过一定阈值的子模块（如 Transformer 的每一层）单独应用 FSDP。这样可以让 All-Gather 操作在更细的粒度上进行，进一步提升显存利用率和计算效率。
- CPU 卸载 (CPU Offload)

当显存依然紧张时，可以开启 CPU 卸载功能。这会将模型参数、梯度或优化器状态的一部分卸载到 CPU 内存中，进一步节省 GPU 显存。但需要注意，这会因 PCIe 带宽限制带来一定的性能开销。

- 混合精度训练 (Mixed Precision)

FSDP 与混合精度训练（如 FP16/BF16）结合使用效果更佳。这不仅能进一步减少显存占用，还能利用 GPU 的 Tensor Core 加速计算。PyTorch FSDP 提供了灵活的配置，可以为参数、梯度聚合等不同操作指定不同的数据类型（例如，参数用 FP16，梯度聚合用 FP32 以保证稳定性）。

- 正确保存模型

由于模型参数是分片存储的，直接保存 FSDP 模型会丢失信息。必须使用 FSDP 提供的 state_dict_type 上下文管理器，将分片的参数重新聚合为完整的模型状态后再进行保存。

In [ ]:
# 正确保存 FSDP 模型的示例代码
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP, StateDictType

# ... 训练代码 ...

# 保存检查点
with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT):
    torch.save(model.state_dict(), "checkpoint.pt")